In [ ]:
from platform import python_version
print(python_version())

### Calculating DEGs statistics

### For each LFC/FDR cutoff set, we get a different set of DEGs
  - LFC: LFC cutoff and FDR_LFC cutoff
  - Pathway: fdr and pval pathway cutoff and min num of genes

### Up and Down DEGs simulation
  - Up and Down DEGs/DAPs
  - Up and Down in pathways

### there are 2 statistical tables
  - pval/fdr cutoff x degs
  - pval/fdr/geneset/quantile degs_in_pathway, num_pathways

In [ ]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"
ROOT_COLAB = ROOT0 / "colab"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import *
from libs.MTD_lib import *
from libs.dashcyto_lib import DASH_CYTO
from libs.calc_degs_lib import CALC_DEGS

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

In [ ]:
root0 = Path(dic_yml['root0'])
root0_data = Path(dic_yml['root0_data'])
root_colab = root0 / 'colab'

email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"


root_project = root0_data / s_project
PSI_ID = 'TCGA-BRCA'
disease = PSI_ID

root_disease = root_project / PSI_ID

print(">>> root0", root0)
print(">>> root_colab", root_colab)
print(">>> root0_data", root0_data)
print(">>> root_project", root_project)
print(">>> root_disease", root_disease)


In [ ]:
root0 = Path(dic_yml['root0'])
root0_data = Path(dic_yml['root0_data'])
root_colab = root0 / 'colab'

email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"


root_project = root0_data / s_project
PSI_ID = 'TCGA-BRCA'
PSI_ID = 'TCGA-ACC'
PSI_ID = 'TCGA-CESC'

disease = PSI_ID

root_project = create_dir(root0_data, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']

case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=root0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

In [ ]:
s_omics

In [ ]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, root0=root0, root0_data=root0_data,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(mtd.echo_parameters())

In [ ]:
from libs.tcga_gdc_lib import GDC

ROOT_DATA = root0 / "data"

gdc = GDC(ROOT0=ROOT0, ROOT_DATA0=ROOT_DATA)

### Get all programs

In [ ]:
force = False
verbose = True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)

prog_id = "TCGA"

df_psi = gdc.get_primary_sites(prog_id=prog_id, force=False, verbose=verbose)
print(len(df_psi))

df_psi.shape, df_psi.columns

In [ ]:
gdc.set_primary_site(psi_id=PSI_ID, verbose=True)

In [ ]:
verbose=False
df_tumor, df_normal, df_gtex_ctrl = gdc.get_file_expression_both_tumor_and_normal(verbose=verbose)

In [ ]:
# gdc.df_meta["SMTSD"].unique()

In [ ]:
print(len(df_tumor.columns)-3)
df_tumor.head(3)

In [ ]:
print(len(df_normal.columns)-3)
df_normal.head(3)

In [ ]:
print(len(df_gtex_ctrl.columns)-2)
df_gtex_ctrl.head(3)

### Check main cols

In [ ]:
mtd.check_lfc_names(verbose=True)

In [ ]:
fname, fname_cutoff = mtd.set_enrichment_name()
fname, os.path.exists(os.path.join(mtd.root_enrich, fname)), fname_cutoff, os.path.exists(os.path.join(mtd.root_enrich, fname_cutoff))

In [ ]:
try:
    dflfc_ori = mtd.dflfc_ori
    print(len(dflfc_ori))
except:
    dflfc_ori = pd.DataFrame()
    
dflfc_ori.head(3)

In [ ]:
lista = ['lncRNA', 'LNC']
dflfc_lnc = dflfc_ori[dflfc_ori.biotype.isin(lista)]
print(f"There are {len(dflfc_lnc)} LNCs\n")
dflfc_lnc.tail(3)

In [ ]:
dflfc_ori = mtd.dflfc_ori
print(len(dflfc_ori))
dflfc_ori_symb = dflfc_ori[~pd.isnull(dflfc_ori.symbol)]
print(len(dflfc_ori_symb))

### Unique symbols

In [ ]:
try:
    symbols = np.unique(dflfc_ori.symbol)
except:
    symbols = []
    
len(symbols), len(dflfc_ori)

In [ ]:
try:
    dflfc = mtd.dflfc
    print(len(dflfc))
except:
    dflfc = pd.DataFrame()
    
dflfc.head(3)

In [ ]:
dfbest = mtd.cfg.open_best_ptw_cutoff()
dfbest

In [ ]:
want_see_best_cutoff = False

if want_see_best_cutoff:
    dfbest = mtd.cfg.dfbest_cutoffs
else:
    dfbest = pd.DataFrame()
dfbest    

In [ ]:
if want_see_best_cutoff:
    dfbest = mtd.cfg.dfbest_cutoffs
    dfa = dfbest[(dfbest.case == case) & (dfbest.normalization == mtd.normalization) & (dfbest.geneset_num == geneset_num) ]
else:
    dfa = pd.DataFrame()

dfa

### Minimum LFC cutoff

In [ ]:
np.log2(1.4)

### DEGs simulation: no DEG/DAPs per cases
### Saving simulation file dfsim in config:
  - all_lfc_cutoffs_taubate_covid19.tsv

#### Sampling

### Cutoff sets to generate the statistical data
  - inf lfc cutoff: 0.40 --> 0.48 ~ 40% modulation  --> 0.65
  - sup fdr cutoff: 0.75 --> no more than

In [ ]:
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
mtd.lfc_list = lfc_list
lfc_list[-1] = 0.0
lfc_list

In [ ]:
fdr_list = np.arange(0.05, 0.76, .01)
mtd.fdr_list = fdr_list
fdr_list

In [ ]:
cutoff_list = np.round([(x, y) for x in lfc_list for y in fdr_list],3)
print(len(cutoff_list))
cutoff_list[:5], cutoff_list[-5:]

### Saving simulations: calc_degs_cutoff_simulation()

  - config/all_lfc_cutoffs_medulloblastoma.tsv
  - while looping in case_list -> save_file -> save txt files

In [ ]:
cutoff_list

In [ ]:
verbose=False
force=False

dfsim = mtd.calc_degs_cutoff_simulation(cutoff_list=cutoff_list, force=force, save_file=force, n_echo=-1, verbose=verbose)

print(dfsim.columns)
print(len(dfsim))

In [ ]:
dfsim = dfsim.sort_values(['case', 'lfc_FDR_cut', 'LFC_cut'], ascending=[False, True, False])
dfsim.head(3)

### Does the simulation worked?

In [ ]:
dfsim = mtd.open_simulation_table()
print(len(dfsim))

dfsim2 = dfsim[dfsim.case == case]
dfsim2.head(3).T

In [ ]:
LFC_cut = 1
lfc_FDR_cut = 0.05

# (dfsim.case == case) &
dfsim[ (dfsim.LFC_cut == LFC_cut) & (dfsim.lfc_FDR_cut == lfc_FDR_cut)].T

In [ ]:
np.unique(dfsim.LFC_cut)

In [ ]:
dfsim.LFC_cut.min(), dfsim.LFC_cut.max()

In [ ]:
np.unique(dfsim.lfc_FDR_cut)

In [ ]:
dfsim.lfc_FDR_cut.min(), dfsim.lfc_FDR_cut.max(), 

In [ ]:
dfsim.LFC_cut.min(), dfsim.LFC_cut.max(), 

In [ ]:
dfsim.lfc_FDR_cut.min(), dfsim.lfc_FDR_cut.max(), 

### Simulations

In [ ]:
case_list

In [ ]:
case = 'Tumor'

mtd.open_case(case, verbose=False)

fname, fname_ori, title = mtd.set_lfc_names()
print(f"fname '{fname}' and title '{title}'")
print(f"LFC cutoff: lfc={mtd.LFC_cut:.3f} fdr={mtd.lfc_FDR_cut}")

print("")
print(mtd.echo_parameters())

### DEGs/DAPs frequency
### Not Normalized

In [ ]:
#dfsim = pdreadcsv( mtd.cfg.fname_lfc_cutoff, mtd.cfg.root_config)
dfsim = mtd.cfg.open_all_lfc_cutoff()
print(len(dfsim))
dfsim.tail(2)

In [ ]:
i=0
case = case_list[i]
print(">>>", case)
df2 = dfsim[dfsim.case == case].copy()
print(len(df2))
df2.head(2)

In [ ]:
dfsim = mtd.open_simulation_table()
print(len(dfsim))
dfsim.head(2)

## Calc all Spearman Correlations - filter the 5 best not repeated fdrs
#### Plot abs_LFC x num of DEG/DAPs
#### corr_cutoff = -.90
#### calc corelation with mtd.LFC_cut_inf = 0.40

In [ ]:
want_calc = False
corr_cutoff=-0.90
nregs_fdr = 5

force=False
verbose=False

df_all_fdr = mtd.calc_all_LFC_FDR_cutoffs(cols2=['n_degs', 'LFC_cut'], corr_cutoff=corr_cutoff, nregs_fdr=nregs_fdr, force=force, verbose=verbose)

In [ ]:
df_all_fdr.head(2)

In [ ]:
print(len(df_all_fdr))
print(df_all_fdr.columns)

df_all_fdr[~pd.isnull(df_all_fdr['corr']) &  (df_all_fdr['corr'] <= -0.9)].head(2)

In [ ]:
df_all_fdr[~pd.isna(df_all_fdr['corr']) &  (df_all_fdr['corr'] <= -0.9)]['corr'].min()

### Plot all dfsim

In [ ]:
# !pip3 install -U kaleido

In [ ]:
fdr_list

In [ ]:
for case in case_list:
    print(">>>", case)
    dic_fig = mtd.plot_all_dfsim(dfsim, case=case, fdr_list=fdr_list, width=1100, height=700, title=None, verbose=force)
        
    for key, fig in dic_fig.items():
        print("\t", key)
        fig.show()
        break # remove to see Up and Dw

    print("\n")
    

In [ ]:
# dfsim.columnscutoff_list

### Restricting the best fdr by Spearman's Correlation

### Must calc for each LFC_cut_inf

In [ ]:
corr_cutoff=-0.90
nregs_fdr = 10
mtd.LFC_cut_inf = 0

verbose=False
force = False

'''
    calc_all_LFC_FDR_cutoffs:
        for case_list
            call calc_nDEG_curve_per_LFC_FDR()
'''
df_all_fdr = mtd.calc_all_LFC_FDR_cutoffs(cols2=['n_degs', 'LFC_cut'], corr_cutoff=corr_cutoff, nregs_fdr=nregs_fdr,
                                          force=force, verbose=verbose)
print(len(df_all_fdr))

### LFC_cut_inf = 0.4

In [ ]:
LFC_cut_inf

In [ ]:
dfsim = mtd.dfsim[mtd.dfsim.case == case]
dfsim = dfsim.sort_values(['lfc_FDR_cut', 'LFC_cut'], ascending=[True, False])

dfsim.lfc_FDR_cut.unique(), dfsim.LFC_cut.unique()

### For FDR == 0.05 (default cutoff) - there is no correlation, is a horizontal flat line for 0.05

In [ ]:
mtd.LFC_cut_inf

In [ ]:
fdr = 0.05

i=0
case = case_list[i]
print(">>>", case)

dfsim2 = dfsim[ (dfsim.case==case) & (dfsim.lfc_FDR_cut == fdr) & (dfsim.LFC_cut >= mtd.LFC_cut_inf) ]
len(dfsim2)

In [ ]:
cols2=['case', 'n_degs', 'lfc_FDR_cut', 'LFC_cut']
dfsim2[cols2].head(5)

In [ ]:
dfsim2[cols2].tail(5)

In [ ]:
dfsim3 = dfsim2.reset_index()

cols3=['index', 'n_degs', 'lfc_FDR_cut', 'LFC_cut']

dfsim3[cols3].head(2)

In [ ]:
dfsim3[cols3].iloc[:, [0,1]].head(3)

In [ ]:
method='spearman'
corr = dfsim3[cols3].iloc[:, [0,1]].corr(method=method)
corr

In [ ]:
pd.isnull(corr)

### mtd.LFC_cut_inf = 0.40

In [ ]:
nregs_fdr = 10

LFC_cut_inf = dic_yml['LFC_cut_inf']
mtd.LFC_cut_inf = LFC_cut_inf

verbose=False
force = False

'''
    calc_all_LFC_FDR_cutoffs:
        for case_list
            call calc_nDEG_curve_per_LFC_FDR()
'''
df_all_fdr = mtd.calc_all_LFC_FDR_cutoffs(cols2=['n_degs', 'LFC_cut'], corr_cutoff=corr_cutoff, nregs_fdr=nregs_fdr,
                                          force=force, verbose=verbose)
print(len(df_all_fdr))

### WNT - Spearman

In [ ]:
i=0
case = case_list[i]

df2 = df_all_fdr[ (df_all_fdr.case == case) ]
print(len(df2))

df2 = df_all_fdr[ (df_all_fdr.case == case) & ( pd.notnull(df_all_fdr['corr'])  ) ]
print(len(df2))
print(f"{mtd.s_deg_dap} max: {df2.n_degs_max.max()}")
df2.head(2)

### G4 Spearman

In [ ]:
for case in case_list:
    df2 = df_all_fdr[ (df_all_fdr.case == case) & ( pd.notnull(df_all_fdr['corr'])  ) ]
    print(f"case: {case:4} n={len(df2)} max {mtd.s_deg_dap}: {df2.n_degs_max.max()}")

### Plot abs_LFC x num of DEGs/DAPs
  - set LFC_cut_inf

In [ ]:
corr_cutoff, nregs_fdr, case_list

In [ ]:
i=0
case  = case_list[i]
print(">>>", case)

cols2=['n_degs', 'LFC_cut']
limit_fdr = -1
method='spearman'

ret, dic_return = mtd.calc_nDEG_curve_per_LFC_FDR(case=case, cols2=cols2, 
                                                  corr_cutoff=corr_cutoff, nregs_fdr=nregs_fdr,
                                                  method=method, verbose=verbose)

len(dic_return)

In [ ]:
list(dic_return.keys())

In [ ]:
len(dic_return['df_fdr'])

In [ ]:
len(dic_return['name_list']), dic_return['name_list']

In [ ]:
len(dic_return['fdrs']), np.array(dic_return['fdrs'])

In [ ]:
df_fdr = dic_return['df_fdr']
df_fdr.head(2)

In [ ]:
LFC_cut_inf2 = mtd.LFC_cut_inf
corr_cutoff

In [ ]:
mtd.LFC_cut_inf = 0.
verbose = False

i=0
case = case_list[i]
mtd.open_case(case)
print(">>>", case)

ret, dic_fig, df_fdr = mtd.plot_nDEG_curve_per_LFC_FDR(case, width=1100, height=700, title=None, 
                                                       corr_cutoff=corr_cutoff, nregs_fdr=nregs_fdr, verbose=verbose)

for key, fig in dic_fig.items():
    print(key)
    fig.show()

## LFC_cut_inf = 0.4 - see the curve

### Ploting only Spearman's limiar curves

In [ ]:

dic_fig = mtd.plot_all_LFC_FDR_cutoffs(width=1100, height=700, title=None, 
                                       corr_cutoff=corr_cutoff, nregs_fdr=nregs_fdr, verbose=force)

for case in case_list:
    print(">>>", case)
    try:
        dic2 = dic_fig[case]
    except:
        continue
        
    for key, fig in dic2.items():
        fig.show()
        # do not show Up and Down
        break
    print("")

In [ ]:
LFC_cut_inf = mtd.LFC_cut_inf
LFC_cut_inf

In [ ]:
case = case_list[0]

df_all_fdr = mtd.open_fdr_lfc_correlation(case, LFC_cut_inf)
df2 = df_all_fdr[ pd.notnull(df_all_fdr['corr']) ]
print(len(df2))
df2.head(6)

### Summary DEG/DAPs + Up and Down (pre-best cutoff)

In [ ]:
verbose=False
per_biotype= False
ensembl = False

dfa = mtd.summary_degs_up_down(per_biotype=per_biotype, ensembl=ensembl, verbose=verbose)
print(len(dfa))
dfa

### per_biotype

In [ ]:
verbose=False
per_biotype= True
ensembl = False

dfa = mtd.summary_degs_up_down(per_biotype=per_biotype, ensembl=ensembl, verbose=verbose)
print(len(dfa))
dfa

### Barplot per per_biotype

In [ ]:
per_biotype = True
ensembl = True
before_best_cutoff = True
fig, dfa = mtd.barplot_up_down_genes_per_case(per_biotype=per_biotype, ensembl=ensembl, before_best_cutoff=before_best_cutoff, width=1100, height=700, verbose=False)
if fig: fig.show()

In [ ]:
width = 1300
          
fig = mtd.plot_all_degs_up_down_per_cutoffs(width=width, height=600, title='', y_anchor=1.05, 
                                            color_up='darkred', color_dw='navy', plot_bgcolor='whitesmoke', verbose=False)
if fig: fig.show()

### Simple summary

In [ ]:
dfa = mtd.summary_degs_up_down(per_biotype=False, ensembl=False, verbose=False)
dfa

### per Biotype

In [ ]:
dfa = mtd.summary_degs_up_down(per_biotype=True, ensembl=False, verbose=False)
dfa

In [ ]:
dfa = mtd.summary_degs_up_down(per_biotype=True, ensembl=True, verbose=False)
dfa

### Review data

In [ ]:
want_review_data = True

if want_review_data:
    i=0
    case = case_list[i]
    mtd.open_case(case, verbose=False)
    
    fname, fname_ori, title = mtd.set_lfc_names()
    print(f"fname '{fname}' and title '{title}'")
    print(f"LFC cutoff: lfc={mtd.LFC_cut:.3f} fdr={mtd.lfc_FDR_cut}")
    
    print("")
    print(mtd.echo_parameters())

### LNCs

In [ ]:
lista = ['lncRNA', 'LNC']
dflfc_lnc = dflfc_ori[dflfc_ori.biotype.isin(lista)]
print(len(dflfc_lnc))
dflfc_lnc.tail(3)

In [ ]:
np.unique(dflfc_lnc.biotype)